In [1]:
import sys
sys.path.append('../src')
from should_be_stdlib import *
from data import *

Operating with: 082620_355l


In [2]:
from itertools import combinations_with_replacement

In [3]:
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt
import numpy as np

In [4]:
# inverse (fast) fourier transform
from scipy.fft import ifft

In [5]:
# euclidean distance
def euclidean(a, b):
    return np.linalg.norm(a - b)

In [6]:
# state fidelity
def fidelity(a, b):
    ra = a / np.linalg.norm(a, ord=1)
    rb = b / np.linalg.norm(a, ord=1)
    return np.sum(np.sqrt(ra * rb)) ** 2

In [7]:
record = load_set()
tuning_curves = get_tc(record)
tuning_curves_rescaled = pd.read_csv(datapath('data_tuning-curves_rescaled.csv'), index_col=0)
tuning_curves_resampled = pd.read_csv(datapath('data_tuning-curves_resampled.csv'), index_col=0)

Loading: 082620_355l


In [8]:
def get_score(a_b):
    a, b = a_b
    a1, b1 = tuning_curves_rescaled.loc[a], tuning_curves_rescaled.loc[b]
    a2, b2 = tuning_curves_resampled.loc[a], tuning_curves_resampled.loc[b]
    return [
        a, b,
        euclidean(a1, b1),
        euclidean(ifft(a2), ifft(b2)),
        fidelity(a1, b1)
    ]

In [9]:
def get_scores():
    pairs = combinations_with_replacement(tuning_curves.index, 2)
    pairs_len = len(tuning_curves) * (len(tuning_curves) + 1) // 2

    from multiprocessing import Pool, cpu_count
    with Pool(processes=cpu_count()) as pool:
        ab = list(tqdm(pool.imap(get_score, pairs), total=pairs_len))

    return pd.DataFrame(
        ab + [[b,a,_1,_2,_3] for (a,b,_1,_2,_3) in ab if a != b],
        columns = ['A', 'B', 'euclidean', 'euclidean_ifft', 'classical_fidelity']
    )

In [11]:
distances = get_scores()

  0%|          | 0/283128 [00:00<?, ?it/s]

In [10]:
corrs_df = tuning_curves.T.corr(method='pearson')
corrs_df.to_csv(datapath('results_correlation-pearson.csv'))
corrs_df

,0,1,2,3,4,5,6,7,8,9,...,742,743,744,745,746,747,748,749,750,751
0,1.000000,0.164669,0.420405,0.492123,0.241726,0.342849,0.527793,0.712033,0.027739,0.752167,...,0.777924,0.666083,0.325484,0.532953,0.561848,-0.462142,0.574119,0.895257,0.808405,-0.331261
1,0.164669,1.000000,0.555532,0.653508,0.381152,0.800599,0.430928,0.159038,0.614715,0.157350,...,0.522150,0.256637,0.768662,0.361594,0.374925,-0.446630,0.642665,0.254006,0.394119,0.473682
2,0.420405,0.555532,1.000000,0.763354,0.722480,0.360215,0.269111,0.118448,0.461793,0.681567,...,0.755896,0.240405,0.793198,0.659275,0.689704,-0.581843,0.516056,0.625686,0.661663,0.090741
3,0.492123,0.653508,0.763354,1.000000,0.689259,0.721893,0.408607,0.504259,0.441169,0.571142,...,0.901470,0.573520,0.771060,0.733407,0.443501,-0.760318,0.684076,0.643272,0.475442,-0.040737
4,0.241726,0.381152,0.722480,0.689259,1.000000,0.473482,0.289666,0.370632,0.573970,0.361766,...,0.658417,0.349777,0.604258,0.801917,0.672568,-0.438427,0.192740,0.342906,0.310907,-0.318479
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
747,-0.462142,-0.446630,-0.581843,-0.760318,-0.438427,-0.481098,-0.178658,-0.242995,0.014339,-0.435692,...,-0.689039,-0.266698,-0.297765,-0.731638,-0.479606,1.000000,-0.553695,-0.732549,-0.484397,-0.289711
748,0.574119,0.642665,0.516056,0.684076,0.192740,0.749281,0.340442,0.278731,0.010117,0.711688,...,0.706102,0.706037,0.525835,0.209585,0.215635,-0.553695,1.000000,0.692192,0.589147,0.178701
749,0.895257,0.254006,0.625686,0.643272,0.342906,0.372838,0.432070,0.480649,-0.072244,0.831678,...,0.812461,0.588274,0.352975,0.618571,0.565928,-0.732549,0.692192,1.000000,0.859677,-0.082964
750,0.808405,0.394119,0.661663,0.475442,0.310907,0.324513,0.659265,0.387396,0.203979,0.683053,...,0.651780,0.445160,0.492160,0.533688,0.628878,-0.484397,0.589147,0.859677,1.000000,0.043757


In [12]:
# turn into symmetric matrices
euclidean_distance = distances.pivot_table(index='A', columns='B')['euclidean'].astype(np.float64)
euclidean_distance.to_csv(datapath('results_euclidean.csv'))
euclidean_distance

B,0,1,2,3,4,5,6,7,8,9,...,742,743,744,745,746,747,748,749,750,751
A,,,,,,,,,,,,,,,,,,,,,
0,0.000000,0.292926,0.249524,0.272509,0.255850,0.294252,0.226408,0.185531,0.305260,0.381206,...,0.155220,0.200815,0.283161,0.255027,0.207463,0.339499,0.310330,0.121679,0.149000,0.484489
1,0.292926,0.000000,0.196998,0.219318,0.201570,0.159551,0.223962,0.215257,0.171952,0.537393,...,0.196724,0.275207,0.155096,0.278878,0.213241,0.298457,0.291601,0.229108,0.241332,0.304955
2,0.249524,0.196998,0.000000,0.186880,0.147934,0.274668,0.259753,0.228392,0.209057,0.415260,...,0.144849,0.283607,0.148373,0.211207,0.157375,0.321107,0.323676,0.170459,0.184447,0.387484
3,0.272509,0.219318,0.186880,0.000000,0.215237,0.208057,0.281372,0.252290,0.266094,0.435600,...,0.139579,0.249611,0.184949,0.207283,0.263407,0.410745,0.276893,0.222011,0.271572,0.457590
4,0.255850,0.201570,0.147934,0.215237,0.000000,0.235355,0.224227,0.149157,0.158665,0.496270,...,0.150798,0.238011,0.191856,0.181441,0.133730,0.250382,0.378074,0.183265,0.234279,0.415776
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
747,0.339499,0.298457,0.321107,0.410745,0.250382,0.359231,0.281013,0.202184,0.235001,0.608062,...,0.317080,0.316245,0.316190,0.396969,0.276583,0.000000,0.473538,0.292677,0.328772,0.409659
748,0.310330,0.291601,0.323676,0.276893,0.378074,0.250378,0.365295,0.362293,0.419170,0.372605,...,0.275525,0.268014,0.323642,0.419037,0.380206,0.473538,0.000000,0.287457,0.304979,0.462343
749,0.121679,0.229108,0.170459,0.222011,0.183265,0.256323,0.208796,0.150723,0.260560,0.401099,...,0.116125,0.199404,0.240336,0.218399,0.160558,0.292677,0.287457,0.000000,0.122871,0.394767


In [13]:
euclidean_distance_ifft = distances.pivot_table(index='A', columns='B')['euclidean_ifft'].astype(np.float64)
euclidean_distance_ifft.to_csv(datapath('results_euclidean-ifft.csv'))
euclidean_distance_ifft

B,0,1,2,3,4,5,6,7,8,9,...,742,743,744,745,746,747,748,749,750,751
A,,,,,,,,,,,,,,,,,,,,,
0,0.000000,8.619105,6.083351,6.366033,6.225860,12.188578,11.523476,11.879766,6.086399,3.732466,...,6.360710,4.846064,9.042840,5.874692,7.376042,10.989382,5.687782,2.457344,7.177354,14.163930
1,8.619105,0.000000,4.152583,4.215826,3.977339,4.599032,5.174555,5.220182,3.823077,9.971673,...,3.461261,5.223362,2.704765,5.319341,3.405381,5.820565,5.244109,10.406792,3.963647,7.105479
2,6.083351,4.152583,0.000000,2.898835,1.995912,7.954346,7.420867,7.429958,2.840739,6.783243,...,2.336381,4.164994,3.784516,3.167682,2.677323,7.188133,4.797823,7.766915,3.421583,9.924262
3,6.366033,4.215826,2.898835,0.000000,3.002581,7.029407,7.479573,7.470706,3.777562,7.273554,...,2.120936,3.752169,4.285947,2.763792,4.040883,8.339276,4.165372,8.065208,4.551096,10.481189
4,6.225860,3.977339,1.995912,3.002581,0.000000,7.437627,6.983857,6.686781,2.132896,7.571297,...,2.218278,3.555158,3.994639,2.857398,2.441087,6.366519,5.329698,7.910597,3.968457,9.978242
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
747,10.989382,5.820565,7.188133,8.339276,6.366519,7.506977,5.966675,4.437892,6.254780,12.428953,...,6.787705,7.543932,5.796806,8.682481,5.905137,0.000000,9.201156,12.714132,6.658945,8.158079
748,5.687782,5.244109,4.797823,4.165372,5.329698,7.917684,8.766136,9.084886,5.553186,6.136366,...,4.147687,3.546722,6.194913,5.802256,5.833769,9.201156,0.000000,7.389959,5.112125,10.766391
749,2.457344,10.406792,7.766915,8.065208,7.910597,14.120132,13.601032,13.934977,7.753943,4.406205,...,8.381613,6.846677,10.923480,7.571013,9.311781,12.714132,7.389959,0.000000,9.283114,15.577968


In [14]:
classical_fidelity = distances.pivot_table(index='A', columns='B')['classical_fidelity'].astype(np.float64)
classical_fidelity.to_csv(datapath('results_classical-fidelity.csv'))
classical_fidelity

B,0,1,2,3,4,5,6,7,8,9,...,742,743,744,745,746,747,748,749,750,751
A,,,,,,,,,,,,,,,,,,,,,
0,1.000000,0.982052,0.985532,0.981058,0.985480,0.982079,0.989736,0.991984,0.978601,0.972993,...,0.994342,0.991597,0.981041,0.985894,0.990303,0.973638,0.980828,0.996252,0.994705,0.947753
1,0.982052,1.000000,0.992120,0.987562,0.992223,0.994016,0.990200,0.990730,0.993944,0.946306,...,0.992298,0.984811,0.994198,0.983512,0.990942,0.981266,0.981689,0.989316,0.987384,0.977304
2,0.985532,0.992120,1.000000,0.989774,0.994934,0.983264,0.985770,0.988314,0.990187,0.967764,...,0.995025,0.982453,0.994981,0.988522,0.994076,0.977141,0.977506,0.993173,0.992107,0.967621
3,0.981058,0.987562,0.989774,1.000000,0.987107,0.989574,0.980274,0.982984,0.980814,0.962678,...,0.994302,0.984601,0.990075,0.989654,0.981010,0.958175,0.984244,0.986943,0.980274,0.949441
4,0.985480,0.992223,0.994934,0.987107,1.000000,0.987877,0.989565,0.995381,0.994505,0.953300,...,0.994984,0.988064,0.991515,0.991361,0.996006,0.986550,0.970127,0.992625,0.987308,0.962555
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
747,0.973638,0.981266,0.977141,0.958175,0.986550,0.970644,0.982660,0.991198,0.987550,0.924616,...,0.977051,0.977820,0.976623,0.963414,0.983113,1.000000,0.950228,0.980666,0.974789,0.964190
748,0.980828,0.981689,0.977506,0.984244,0.970127,0.987961,0.973582,0.972305,0.961707,0.972538,...,0.985047,0.985317,0.976602,0.966418,0.970018,0.950228,1.000000,0.983049,0.980477,0.950188
749,0.996252,0.989316,0.993173,0.986943,0.992625,0.986106,0.991173,0.994928,0.984257,0.968022,...,0.996763,0.991290,0.985778,0.989752,0.994026,0.980666,0.983049,1.000000,0.995974,0.966393
